# Chunk Happens

Part 7 built the tool: turn a sentence into a vector, using a model that's actually trained for it. Time to use it for what it's for -- a real RAG system, over real documents.

There's a practical wrinkle first, though. Three sentences cost three API calls; a 218-page book cut into passages costs **several hundred**, and you'd re-pay them every time you rebuild the index. Gemini's free tier is generous but not unlimited (Part 2 already ran into its per-day request cap), and burning it on bulk indexing would be a poor trade -- so everything below runs on `multilingual-e5-small`, the local model Part 7 settled on, recreated exactly as it was.

We'll build it in two stages, because that's how it goes in practice:

1. **On a clean document first** -- the IoT book from Part 2, `documents/PLIDO_BOOK_en.pdf`. One well-structured text, so we can see the machinery work without anything else getting in the way.
2. **Then on the open web** -- an agent that goes and collects real TAF documentation into `documents/`, at which point retrieval quality drops noticeably. Diagnosing *why*, and fixing it, is where most of the real work in a RAG system actually lives.

## Chunking: why we can't embed a whole book

`documents/PLIDO_BOOK_en.pdf`, the IoT book Part 2 stuffed *whole* into the agent's instructions, is 218 pages, roughly 75,000 words.

We can't embed it as one vector. Partly because a single vector summarising 75,000 words would be so generic it'd match every question equally, but mostly for a harder reason: the model has a **512-token input limit**, so anything past roughly the first 350 words is silently discarded. Not an error, not a warning -- just truncated, and you'd never know from the output.

So we **chunk**: cut the text into passages small enough to embed intact, each one specific enough to be a meaningful answer on its own. Two parameters matter:

* **Size** -- 180 words here, comfortably inside the 512-token limit even with long technical words. Too big and you hit the truncation trap; too small and a passage loses the context that makes it meaningful.
* **Overlap** -- 40 words repeated between consecutive chunks, so a sentence that happens to straddle a boundary still appears whole in one of them.

One more thing this particular PDF forces on us. A book has a table of contents and an index -- pages of `LoRaWAN . . . . . . . . 27, 59`. Those chunks are almost pure punctuation, they carry no meaning, and they pollute results. So we drop any chunk that isn't mostly letters. Real corpora always need some cleanup like this; the only question is which kind.

In [ ]:
# Program 1: recreate the local embedding model from Part 7, then chunk and index the IoT book

import os
import time
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from pypdf import PdfReader
from dotenv import load_dotenv

load_dotenv(override=True)

embed_name = "intfloat/multilingual-e5-small"
embed_tokenizer = AutoTokenizer.from_pretrained(embed_name)
embed_model = AutoModel.from_pretrained(embed_name)
embed_model.eval()

def embed(texts, batch_size=32):
    """Same mean-pooling as Part 7, plus length-1 normalisation, batched so a few hundred
    passages don't take a few hundred forward passes."""
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = embed_tokenizer(texts[start:start + batch_size], return_tensors="pt",
                                truncation=True, max_length=512, padding=True)
        with torch.no_grad():
            hidden_states = embed_model(**batch).last_hidden_state
        mask = batch["attention_mask"].unsqueeze(-1).float()
        pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)
        vectors.append(F.normalize(pooled, dim=-1))
    return torch.cat(vectors)

def embed_passages(texts):
    return embed([f"passage: {t}" for t in texts])

def embed_query(text):
    return embed([f"query: {text}"])[0]

DOCS_DIR = os.path.abspath(os.path.join(os.getcwd(), "documents"))

book = PdfReader(os.path.join(DOCS_DIR, "PLIDO_BOOK_en.pdf"))
book_text = "\n".join((page.extract_text() or "") for page in book.pages)
print(f"Book: {len(book.pages)} pages, {len(book_text.split()):,} words")

def chunk_text(text, size=180, overlap=40):
    """Cut text into overlapping passages of `size` words."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + size]))
        start += size - overlap
    return chunks

def is_useful(chunk):
    """Drop table-of-contents and index chunks: mostly dots and page numbers, few real words."""
    letters = sum(character.isalpha() for character in chunk)
    return letters / max(len(chunk), 1) > 0.6

raw_chunks = chunk_text(book_text)
book_chunks = [c for c in raw_chunks if is_useful(c)]
print(f"{len(raw_chunks)} chunks -> {len(book_chunks)} kept "
      f"({len(raw_chunks) - len(book_chunks)} dropped as table-of-contents/index)")

start_time = time.time()
book_vectors = embed_passages(book_chunks)
print(f"Indexed in {time.time() - start_time:.0f}s, entirely on this machine -- zero API calls.")

## Retrieving from the index

The index is built. Retrieval is now the cosine similarity from Part 4, applied at scale: embed the question, compare it to every stored chunk, keep the closest few.

Because the vectors are normalised, that whole comparison is a single matrix multiplication -- `vectors @ question`, one dot product per chunk, done in one call rather than a Python loop over hundreds of entries. This is the same operation a real vector database (FAISS, Chroma, pgvector...) optimises for millions of chunks; at our scale, plain PyTorch is entirely enough.

Note we ask for the top **three** chunks, not just the best one. Retrieval isn't perfect, and giving the model a few candidates lets it pick out the relevant part itself -- a cheap and very effective safety margin.

In [ ]:
# Program 2: retrieve the passages closest to a question

def retrieve(question, chunks, vectors, top_k=3):
    question_vector = embed_query(question)
    scores = vectors @ question_vector          # one dot product per chunk, in one operation
    best = scores.topk(top_k)
    return [(scores[i].item(), chunks[i]) for i in best.indices.tolist()]

for question in ["How does 6LoWPAN compress IPv6 headers?",
                 "Qu'est-ce que le protocole MQTT ?"]:
    print(f"Q: {question}")
    for score, chunk in retrieve(question, book_chunks, book_vectors):
        print(f"  {score:.3f}  {' '.join(chunk.split())[:150]}...")
    print()

Both questions land on the right passage, and the second one is worth a second look: the question is in French, the book is in English, and they share almost no vocabulary -- yet the MQTT passage comes back. That's the multilingual embedding doing exactly what Part 7's Program 1.2 found `SmolLM2` couldn't: `multilingual-e5-small`, recreated in Program 1 above, was actually trained for this, now proving it across 400 passages of a real book instead of six toy sentences.

## Wrapping retrieval as a tool

Like every other capability in this course, retrieval becomes useful to an agent once it's wrapped as a tool. This one doesn't fetch a web page or write a file -- it searches the index we just built, and hands back raw passages for the model to answer from.

Compare this with Part 2's `book_agent`, which put the *entire* book into its instructions on every single call. Same book, same questions, but now the model only ever sees the three passages that matter.

In [ ]:
# Program 3: a RAG agent over the book -- it only ever sees the passages it retrieves

from IPython.display import Markdown, display
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel
from openai import AsyncOpenAI

RENNES_BASE_URL = "https://ragarenn.eskemm-numerique.fr/sso/instance@imt/api/"
rennes_client = AsyncOpenAI(base_url=RENNES_BASE_URL, api_key=os.environ["RENNES_API_KEY"])
rennes_model = OpenAIChatCompletionsModel(model="ilaas/mistral-small-4-119b", openai_client=rennes_client)

@function_tool
def search_book(question: str):
    """Search the Internet of Things book for the passages most relevant to a question."""
    passages = retrieve(question, book_chunks, book_vectors)
    return "\n\n---\n\n".join(chunk for _, chunk in passages)

book_rag_agent = Agent(
    name="PLIDO Book RAG Agent",
    instructions="Answer questions about the Internet of Things using the search_book tool. "
                 "Base your answer only on the passages it returns -- if they don't contain "
                 "the answer, say so rather than inventing one.",
    model=rennes_model,
    tools=[search_book],
)

result = await Runner.run(book_rag_agent, "What is 6LoWPAN and why is it needed?", max_turns=6)
display(Markdown(result.final_output))

A correct, grounded answer, built from three passages instead of 75,000 words. That's RAG working exactly as advertised.

It worked this smoothly for a reason worth naming: the book is a *clean* corpus. One document, written by one author, no navigation menus, no duplicated pages, consistent structure throughout. Now let's see what happens when the corpus comes from the open web instead.

## Building a real corpus: the librarian agent

We want documentation about the TAF programs -- and unlike the book, nobody hands us a single tidy PDF. It's scattered across the school's website, brochures, and PDF catalogues.

So we'll do what Part 6 taught us to do: build an agent for it. This one searches with Brave, downloads whatever it finds, extracts plain text (via `pypdf` for PDFs, `BeautifulSoup` for web pages), and saves the result into `documents/`. Two tools, and it decides for itself what's worth keeping.

Note the failure handling in `save_document`. Fetching real documents from the real web fails constantly -- dead links, login walls, pages that turn out to be nearly empty. Rather than crash the agent, every failure returns a *message* the model can read and act on, so it just tries the next result. An agent that can't cope with failing tools is useless outside a demo.

You'll also need your Brave API key from Part 6 (`BRAVE_API_KEY` in `.env`); the free tier covers this comfortably.

In [ ]:
# Program 4: an agent that collects real TAF documentation from the web into documents/

import io
import re
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (PLIDOagent-course/1.0; educational use)"}
BRAVE_API_KEY = os.environ["BRAVE_API_KEY"]

@function_tool
def search_documents(query: str):
    """Search the web with Brave for documents about IMT Atlantique TAF programs.
    Returns up to 5 result titles and URLs."""
    response = requests.get("https://api.search.brave.com/res/v1/web/search",
                            headers={"Accept": "application/json", "X-Subscription-Token": BRAVE_API_KEY},
                            params={"q": query, "count": 5}, timeout=20)
    time.sleep(1.2)  # the free tier allows about one request per second
    if response.status_code != 200:
        return f"Search failed (HTTP {response.status_code}). Try again or rephrase the query."
    results = response.json().get("web", {}).get("results", [])
    return "\n".join(f"{r['title']}: {r['url']}" for r in results) or "No results found."

def extract_text(response):
    """Plain text out of a PDF or a web page.

    For web pages, dropping the navigation matters more than it looks: menus, footers and
    campus lists are repeated on every page of a site, they mention a little of everything,
    and so they end up moderately close to *every* question -- crowding out the passages
    that are genuinely about one topic. So we strip those tags and keep the main content.
    """
    if "pdf" in response.headers.get("content-type", "").lower():
        pdf = PdfReader(io.BytesIO(response.content))
        return "\n".join((page.extract_text() or "") for page in pdf.pages)

    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup(["script", "style", "nav", "header", "footer", "aside", "form"]):
        tag.decompose()
    main = soup.find("main") or soup.find("article") or soup.find(attrs={"role": "main"}) or soup
    return main.get_text("\n", strip=True)

@function_tool
def save_document(url: str, filename: str):
    """Download a document (PDF or web page) and save its plain text into documents/<filename>.txt.
    Use a short descriptive filename, without extension. Returns how many characters were saved,
    or an explanation if the document could not be fetched or held too little text."""
    if not re.fullmatch(r"[A-Za-z0-9_-]+", filename):
        return "Invalid filename: use only letters, digits, dashes and underscores."
    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
    except Exception as error:
        return f"Could not fetch {url}: {error}"
    if response.status_code != 200:
        return f"Could not fetch {url}: HTTP {response.status_code}"
    try:
        text = extract_text(response)
    except Exception as error:
        return f"Could not read the document at {url}: {error}"

    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    if len(text) < 1500:
        # Landing pages that only host a download link end up here, and that is the point:
        # they are almost pure title and file metadata, but they mention the topic often
        # enough to outrank genuine documentation if we let them into the corpus.
        return f"Only {len(text)} characters of real content at {url} -- too thin, skipping."

    with open(os.path.join(DOCS_DIR, f"{filename}.txt"), "w") as f:
        f.write(f"SOURCE: {url}\n\n{text}")
    return f"Saved {len(text)} characters to documents/{filename}.txt"

librarian_instructions = """You build a local documentation library about the TAF (Thematique
d'Approfondissement) programs at IMT Atlantique.

Work autonomously -- never ask the user anything. Then:
1. Use search_documents to find pages and PDFs describing IMT Atlantique's TAF programs: their
   content, their syllabus, and the careers they lead to.
2. For each promising result, call save_document with a short descriptive filename. PDF
   catalogues covering several TAF at once are the most valuable of all.
3. Many results will fail -- dead links, login walls, pages with almost no real content. That
   is expected: just move on to the next result, or try a different search.
4. Stop once you have saved 3 useful documents, and reply with a one-line summary.
"""

librarian = Agent(name="TAF Librarian", instructions=librarian_instructions,
                  model=rennes_model, tools=[search_documents, save_document])

result = await Runner.run(librarian, "Build the TAF documentation library.", max_turns=30)
print(result.final_output)

print("\nFiles now in documents/:")
for filename in sorted(f for f in os.listdir(DOCS_DIR) if f.endswith(".txt")):
    size_kb = os.path.getsize(os.path.join(DOCS_DIR, filename)) / 1024
    print(f"  {filename}  ({size_kb:.0f} KB)")

Open `documents/` and look at what came back. In our run the agent saved the official *TAF 2019* catalogue (27 pages, with a real per-programme breakdown), the engineering-degree brochure, and a couple of web pages -- while several other candidates failed with 404s or login walls, exactly as the instructions anticipated.

Two practical notes:

* **Results vary between runs.** The web changes, and the agent picks different sources each time. `documents/backup/` in this repository holds a snapshot of what we collected, so you have a working corpus even if today's searches come back empty.
* **You can add your own.** Anything you drop into `documents/` as a `.txt` file -- a syllabus, your own course notes -- gets indexed by the next program, with no code change. That's the point of keeping the corpus a plain directory of text files.

## The same code, a much worse result

Now let's index this new corpus with *exactly* the code from Program 1 -- same chunking, same junk filter, same embedding model -- and ask it the kind of question a student would actually ask.

In [ ]:
# Program 5.1: index the collected documents with the very same code -- and watch it struggle

def load_documents(directory, chunker):
    """Read every .txt file in a directory, remembering which file each chunk came from."""
    chunks, sources = [], []
    for filename in sorted(f for f in os.listdir(directory) if f.endswith(".txt")):
        text = open(os.path.join(directory, filename)).read()
        for chunk in chunker(text):
            if is_useful(chunk):
                chunks.append(chunk)
                sources.append(filename)
    return chunks, sources

# Fall back on the committed snapshot if the librarian came home empty-handed.
corpus_dir = DOCS_DIR
if not any(f.endswith(".txt") for f in os.listdir(DOCS_DIR)):
    corpus_dir = os.path.join(DOCS_DIR, "backup")
    print("No documents collected -- falling back on documents/backup/\n")

taf_chunks, taf_sources = load_documents(corpus_dir, chunk_text)
taf_vectors = embed_passages(taf_chunks)
print(f"{len(taf_chunks)} chunks from {len(set(taf_sources))} documents\n")

for question in ["Which TAF is about data science and artificial intelligence?",
                 "I want to work on connected devices and industry 4.0"]:
    print(f"Q: {question}")
    for score, chunk in retrieve(question, taf_chunks, taf_vectors):
        print(f"  {score:.3f}  {' '.join(chunk.split())[:110]}...")
    print()

Compare that with the book. There, each question landed on the passage that answered it. Here the top hits are vague -- a page header, a run of programme acronyms, a chunk about medical imaging when you asked about connected devices. The catalogue really does contain a `DASCI – DATA SCIENCE` record and an `IOT – INTERNET OF THINGS` one, and neither comes back.

Nothing is broken: same model, same code, same junk filter. What changed is the **shape of the documents**.

The book is continuous prose. Cut it anywhere and you still get a passage about one topic, because that's how prose works -- a paragraph about 6LoWPAN is surrounded by more text about 6LoWPAN.

A course catalogue is the opposite: a **list of short, self-contained records**, one per programme, each only a dozen lines long. Cutting every 180 words pays no attention to those boundaries, so one chunk ends up holding the tail of one TAF, all of the next, and the start of a third. Its embedding is the average of three unrelated programmes -- close to nothing in particular -- while a question about exactly one of them has nothing precise to match.

The fix isn't a better model or a bigger chunk. It's to **cut where the document says to cut**: the catalogue marks every record with a heading of its own,

```
CYBER – CYBERSECURITY (4+3)
DASCI – DATA SCIENCE: FROM DATA TO DECISION-MAKER (3+4)
IOT – INTERNET OF THINGS FOR THE INDUSTRY 4.0 (3+3)
```

so we split on those instead, and each chunk becomes exactly one programme. Fixed-size chunking stays the fallback for documents with no such structure -- like the book.

In [ ]:
# Program 5.2: cut on the document's own section headings instead of every 180 words

# A TAF record always starts with a heading like "CYBER – CYBERSECURITY (4+3)":
# an acronym, a dash, a title, and the number of semesters.
SECTION_HEADING = re.compile(r"^\s*[A-Z][A-Z0-9&*\-' ]{1,40}\s*[–-]\s+.{3,70}\(\d\+\d\)\s*$")

def chunk_structured(text):
    """Split on section headings when the document has them, fixed-size otherwise."""
    lines = text.splitlines()
    headings = [i for i, line in enumerate(lines) if SECTION_HEADING.match(line)]

    if len(headings) < 3:            # no real structure -- the book takes this path
        return chunk_text(text)

    chunks = []
    if headings[0] > 0:              # whatever comes before the first heading
        chunks += chunk_text("\n".join(lines[:headings[0]]))
    for start, end in zip(headings, headings[1:] + [len(lines)]):
        section = "\n".join(lines[start:end]).strip()
        # One record per chunk -- unless a record is itself too long to embed intact.
        chunks += chunk_text(section) if len(section.split()) > 180 else [section]
    return chunks

taf_chunks, taf_sources = load_documents(corpus_dir, chunk_structured)
taf_vectors = embed_passages(taf_chunks)
print(f"{len(taf_chunks)} chunks, one per programme wherever the document allowed it\n")

for question in ["Which TAF is about data science and artificial intelligence?",
                 "I want to work on connected devices and industry 4.0"]:
    print(f"Q: {question}")
    for score, chunk in retrieve(question, taf_chunks, taf_vectors):
        print(f"  {score:.3f}  {' '.join(chunk.split())[:110]}...")
    print()

Both questions now land on the right record -- `DASCI – DATA SCIENCE: FROM DATA TO DECISION-MAKER` and `IOT – INTERNET OF THINGS FOR THE INDUSTRY 4.0` -- where Program 5.1 surfaced neither. Same corpus, same embedding model, same retrieval code. **The only thing that changed is where we cut the text**, and it was worth more than any model upgrade would have been. (While writing this part we tried a bigger embedding model: it made no difference. Chunking did.)

Two caveats worth keeping in mind, because they're the normal condition of RAG rather than defects of this example:

* **Your results will differ from ours.** The librarian collects whatever the web offers today, so your corpus isn't ours. A question whose TAF happens to be missing from the documents you collected will still come back with something vaguely related -- retrieval always returns its closest matches, even when nothing is genuinely close. Being unable to say "I don't know" is a real limitation of plain similarity search.
* **The heading pattern is specific to these catalogues.** `SECTION_HEADING` matches how IMT Atlantique formats a TAF record; another corpus needs another rule -- Markdown `##` headings, numbered clauses, `<h2>` tags. There's no universal chunker, which is exactly why chunking deserves this much attention.

Wire `taf_chunks` into a `@function_tool` exactly like `search_book` in Program 3, and you have Part 6's TAF advisor again -- except its knowledge now comes from real documents you can open and check, rather than descriptions written by hand.

## Key takeaways

* **RAG** means embedding source material once, in advance, then using cosine similarity to retrieve only the passages relevant to a question -- instead of stuffing everything into every prompt, which is what Part 2's `book_agent` and Part 6's Program 14 both did.
* Retrieval is a third kind of decision-maker alongside Part 6's algorithmic and agentic ones: relevance decided by nearest-neighbour search over embeddings, not by fixed Python branches and not by an LLM reading everything.
* **Bulk embedding belongs on a local model.** A hosted API is right for occasional calls, but indexing a corpus means hundreds of them, re-paid on every rebuild -- enough to exhaust a free tier for no benefit, which is why everything here reused Part 7's `multilingual-e5-small` instead of Gemini.
* **Every embedding model has a hard input limit** (512 tokens here). Exceed it and your text is silently truncated -- no error, no warning, just half your chunk quietly discarded.
* **Chunking is where RAG quality is won or lost.** The same code that worked on continuous prose failed on a catalogue of short records, because fixed-size cuts ignored the boundaries between them. Cutting on the document's own headings fixed it outright.
* **Extraction quality matters just as much.** A scraped web page is mostly navigation; menus repeated across every page are generic enough to rank against every question. Strip them at extraction time, and thin landing pages fall below the length threshold on their own.
* **Retrieval can't say "I don't know".** It always returns its closest matches, however far away they are -- so a corpus missing the answer produces confident-looking passages about something else.
* When retrieval disappoints, look at your **documents** before reaching for a bigger model. Ours was a chunking problem and an extraction problem -- neither of which a better embedding model would have solved.

## Function reference

A quick reference for the less obvious functions and methods used in this notebook (skipping ones already familiar from earlier parts, like `AutoTokenizer.from_pretrained`, `AutoModel.from_pretrained`, `Agent`, `Runner.run`, and `@function_tool` -- all introduced in Part 7):

| Function (module) | Arguments | Returns | Used in |
|---|---|---|---|
| `vectors @ query` (`torch`) | a matrix of chunk vectors, one query vector | one similarity score per chunk, in a single operation | Programs 2, 5.1, 5.2 |
| `tensor.topk(k)` (`torch`) | how many results to keep | the k highest scores and their indices | Programs 2, 5.1, 5.2 |
| `PdfReader(path_or_bytes)` (`pypdf`) | a file path, or a `BytesIO` of downloaded bytes | a PDF whose `.pages` expose `.extract_text()` | Programs 1, 4 |
| `soup.find("main")` (`bs4`) | a tag name | the first matching element, or `None` -- used to skip navigation | Program 4 |